In [91]:
import torch

In [92]:
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(torch.cuda.get_device_properties(i).name)
else:
    print('GPU not available!')

NVIDIA GeForce RTX 3060 Laptop GPU


In [93]:
import json

with open('../data/saved_rivens/syam.json') as file:
    data = json.load(file)
    
data

{'itemname': 'syam',
 'orders': [{'type': 'sell',
   'user': {'status': 'offline'},
   'item': {'weapon_url_name': 'syam',
    'polarity': 'vazarin',
    'type': 'riven',
    'attributes': [{'value': 15.9,
      'positive': True,
      'url_name': 'channeling_damage'},
     {'value': 68.6, 'positive': True, 'url_name': 'critical_damage'},
     {'value': 133.2, 'positive': True, 'url_name': 'critical_chance'},
     {'value': -68.8, 'positive': False, 'url_name': 'puncture_damage'}],
    'mod_rank': 8,
    'name': 'para-acricron',
    're_rolls': 33,
    'mastery_level': 12},
   'platinum': 888888},
  {'type': 'sell',
   'user': {'status': 'offline'},
   'item': {'name': 'croni-loctitis',
    're_rolls': 0,
    'mastery_level': 14,
    'polarity': 'naramon',
    'attributes': [{'value': 63.6,
      'positive': True,
      'url_name': 'critical_damage'},
     {'value': 1.4, 'positive': True, 'url_name': 'range'},
     {'value': 40.5, 'positive': True, 'url_name': 'fire_rate_/_attack_speed

In [94]:
orders = data['orders']
attributes_list = [order['item']['attributes'] for order in orders]
flatten_attributes = [attribute for attribute_list in attributes_list for attribute in attribute_list]
attribute_names = {attribute['url_name'] for attribute in flatten_attributes}

attribute_names

{'base_damage_/_melee_damage',
 'chance_to_gain_combo_count',
 'chance_to_gain_extra_combo_count',
 'channeling_damage',
 'channeling_efficiency',
 'cold_damage',
 'combo_duration',
 'critical_chance',
 'critical_chance_on_slide_attack',
 'critical_damage',
 'damage_vs_corpus',
 'damage_vs_grineer',
 'damage_vs_infested',
 'electric_damage',
 'finisher_damage',
 'fire_rate_/_attack_speed',
 'heat_damage',
 'puncture_damage',
 'range',
 'slash_damage',
 'status_chance',
 'status_duration',
 'toxin_damage'}

In [95]:
def parse_order(order):
    item_attributes = {attribute['url_name']: attribute['value'] for attribute in order['item']['attributes']}
    price = order['platinum']
    attribute_values = {}
    for attribute_name in attribute_names:
        attribute_values[attribute_name] = item_attributes.get(attribute_name) or 0
    
    return attribute_values, price

In [96]:
parse_order(orders[0])

({'chance_to_gain_extra_combo_count': 0,
  'damage_vs_corpus': 0,
  'damage_vs_grineer': 0,
  'puncture_damage': -68.8,
  'critical_chance_on_slide_attack': 0,
  'electric_damage': 0,
  'status_duration': 0,
  'channeling_damage': 15.9,
  'base_damage_/_melee_damage': 0,
  'critical_damage': 68.6,
  'range': 0,
  'chance_to_gain_combo_count': 0,
  'channeling_efficiency': 0,
  'slash_damage': 0,
  'combo_duration': 0,
  'fire_rate_/_attack_speed': 0,
  'critical_chance': 133.2,
  'damage_vs_infested': 0,
  'toxin_damage': 0,
  'cold_damage': 0,
  'status_chance': 0,
  'heat_damage': 0,
  'finisher_damage': 0},
 888888)

In [97]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [119]:
X = [list(parse_order(order)[0].values()) for order in orders]
y = [parse_order(order)[1] for order in orders]

print(X)
print(y)

print(f'[Sanity Check] Number of plat prices matches number of rivens: {len(X) == len(y)}')

[[0, 0, 0, -68.8, 0, 0, 0, 15.9, 0, 68.6, 0, 0, 0, 0, 0, 0, 133.2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 63.6, 1.4, 0, -42.4, 0, 0, 40.5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, -71.5, 0, 0, 0, 0, 120.9, 65.0, 0, 0, 0, 0, 0, 0, 133.3, 0, 0, 0, 0, 0, 0], [0, 0, 0, -61.0, 0, 0, 0, 0, 112.6, 63.1, 0, 0, 0, 0, 0, 0, 122.3, 0, 0, 0, 0, 0, 0], [0, 0, 0, -62.5, 0, 0, 0, 0, 0, 61.5, 0, 0, 0, 0, 0, 39.4, 118.7, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 115.0, 62.2, 0, 0, -38.3, 0, 0, 0, 133.3, 0, 0, 0, 0, 0, 0], [0, 0, 1.32, 0, 0, 0, 0, 0, 0, 60.2, 0, 0, 0, 0, 0, 0, 123.1, 0, 0, 0, 0, 0, -61.0], [0, 0, 0, 0, 0, 0, 0, 0, 105.0, 61.0, 0, 0, -43.0, 0, 0, 0, 137.0, 0, 0, 0, 0, 0, 0], [0, 0, 1.29, 0, 0, 0, 0, 0, 119.5, 62.6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -64.4], [0, 0, 0, 0, 0, 0, 0, 0, 115.0, 62.2, 0, 0, -38.3, 0, 0, 0, 133.3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 119.0, 69.2, 0, 0, 0, 0, -4.2, 0, 118.9, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 104.4, 61.2, 0, 0, 0, -60.9, 0, 0, 1

In [120]:
from torch.utils.data import Dataset


class RivenDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [121]:
from torch import nn

number_of_attributes = len(attribute_names)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.hidden_layer1 = nn.Linear(number_of_attributes, 64)
        self.hidden_layer2 = nn.Linear(64, 32)
        self.hidden_layer3 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, x):
        x = torch.relu(self.hidden_layer1(x))
        x = torch.relu(self.hidden_layer2(x))
        x = torch.relu(self.hidden_layer3(x))
        x = self.output(x)
        return x.squeeze(-1)

In [122]:
from torch.utils.data import DataLoader, random_split, TensorDataset


X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y, dtype=torch.float32).to(device)

dataset = TensorDataset(X_tensor, y_tensor)

training_data, test_data = random_split(dataset, [0.8, 0.2])

train_dataloader = DataLoader(training_data, shuffle=True)
test_dataloader = DataLoader(test_data, shuffle=True)

In [137]:
from torch import optim

model = NeuralNetwork().to(device)
print(model)

loss_function = nn.MSELoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-2)

NeuralNetwork(
  (hidden_layer1): Linear(in_features=23, out_features=64, bias=True)
  (hidden_layer2): Linear(in_features=64, out_features=32, bias=True)
  (hidden_layer3): Linear(in_features=32, out_features=16, bias=True)
  (output): Linear(in_features=16, out_features=1, bias=True)
)


In [138]:
num_epochs = 300
epoch = 0

loss_values = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_dataloader:
        y_pred = model(X_batch)
        loss = loss_function(y_pred, y_batch.type(torch.float32))
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
     
    epoch_loss = running_loss / len(training_data.dataset)   
    loss_values.append(epoch_loss)
    print(f'[Epoch {epoch}] MSE: {epoch_loss:.4f}')


[Epoch 0] MSE: 1723648609.6884
[Epoch 1] MSE: 1721467025.8172
[Epoch 2] MSE: 1723642550.0213
[Epoch 3] MSE: 1719786991.5308
[Epoch 4] MSE: 1716759003.4273
[Epoch 5] MSE: 1698275375.1280
[Epoch 6] MSE: 1708262214.0867
[Epoch 7] MSE: 1742202568.9804
[Epoch 8] MSE: 1691129656.3924
[Epoch 9] MSE: 1721018715.9836
[Epoch 10] MSE: 1680803451.9238
[Epoch 11] MSE: 1680391925.4427
[Epoch 12] MSE: 1689450011.7659
[Epoch 13] MSE: 1644451546.9427
[Epoch 14] MSE: 1631827826.9767
[Epoch 15] MSE: 1680480407.4672
[Epoch 16] MSE: 1569862996.6390
[Epoch 17] MSE: 1463140709.4275
[Epoch 18] MSE: 1641849387.6943
[Epoch 19] MSE: 1522186448.3012
[Epoch 20] MSE: 1493953725.4292
[Epoch 21] MSE: 1533896113.8895
[Epoch 22] MSE: 1450358929.7992
[Epoch 23] MSE: 1453381984.2185
[Epoch 24] MSE: 1536108716.8892
[Epoch 25] MSE: 1725865320.7193
[Epoch 26] MSE: 1363278343.8516
[Epoch 27] MSE: 1504214511.1558
[Epoch 28] MSE: 1312133373.3726
[Epoch 29] MSE: 1462829183.7045
[Epoch 30] MSE: 1424516461.3813
[Epoch 31] MSE: 13

In [51]:
import pathlib

model_save_path = pathlib.Path('../data/model.dat')
torch.save(model, model_save_path)

In [ ]:
model = NeuralNetwork()
model.load_state_dict(model_save_path)

In [55]:
print(test_data[0])

(tensor([  0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,  56.2000,   0.0000,
          0.0000,  36.1000, 137.5000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000, -70.0000], device='cuda:0'), tensor([600], device='cuda:0'))


In [142]:
model.eval()
with torch.no_grad():
    running_loss = 0.0
    for X_batch, y_batch in test_dataloader:
        y_pred = model(X_batch)
        loss = loss_function(y_pred, y_batch.type(torch.float32))
        print(y_pred)
        print(y_batch)
        running_loss += loss.item() * X_batch.size(0)
     
    loss = running_loss / len(training_data.dataset)
    print(f'Validation MSE: {loss:.4f}')

tensor([918.7840], device='cuda:0')
tensor([300.], device='cuda:0')
tensor([1223.1305], device='cuda:0')
tensor([500.], device='cuda:0')
tensor([203.5054], device='cuda:0')
tensor([450.], device='cuda:0')
tensor([1596.5173], device='cuda:0')
tensor([666.], device='cuda:0')
tensor([845.4483], device='cuda:0')
tensor([250.], device='cuda:0')
tensor([91.9066], device='cuda:0')
tensor([75.], device='cuda:0')
tensor([1215.3423], device='cuda:0')
tensor([450.], device='cuda:0')
tensor([14.3022], device='cuda:0')
tensor([620.], device='cuda:0')
tensor([1314.2247], device='cuda:0')
tensor([1000.], device='cuda:0')
tensor([906.9947], device='cuda:0')
tensor([800.], device='cuda:0')
tensor([802.9932], device='cuda:0')
tensor([950.], device='cuda:0')
tensor([319.5737], device='cuda:0')
tensor([500.], device='cuda:0')
tensor([931.1423], device='cuda:0')
tensor([400.], device='cuda:0')
tensor([644.4636], device='cuda:0')
tensor([179.], device='cuda:0')
tensor([1097.3953], device='cuda:0')
tensor([1

In [ ]:
# TODO include rank as a dimension